<a href="https://colab.research.google.com/github/MuskaanPrabhakar/ExamScope/blob/main/ExamScope.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ExamScope AI — installing and importing

In [ ]:
!pip install langchain-community langchain-text-splitters langchain-huggingface langchain_chroma

In [ ]:
!pip install langchain_pymupdf4llm

In [ ]:
import pymupdf4llm

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

In [ ]:
headers_to_split_on = [
("#", "Header 1"),
("##", "Header 2"),
("###", "Header 3"),
("####", "Header 4")
]

In [ ]:
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

In [ ]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,   # max characters per chunk
    chunk_overlap=50 # overlap between chunks
)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
from langchain_chroma import Chroma

In [ ]:
import hdbscan
from collections import defaultdict

In [ ]:
import re

In [ ]:
import gradio as gr

In [ ]:
!pip install openai
from openai import OpenAI

In [ ]:
import time

In [ ]:
from google.colab import userdata

In [ ]:
#for using only gemini models
client1 = OpenAI(
    base_url ="https://generativelanguage.googleapis.com/v1beta/",
    api_key=userdata.get("GEMINI_API_KEY")
)

In [ ]:
#for using only gemini models
client2 = OpenAI(
    base_url ="https://generativelanguage.googleapis.com/v1beta/",
    api_key=userdata.get("GEMINI2")
)

In [ ]:
import json
import os


In [ ]:
vectorstore = None
all_chunks = None
embedding_model = None
embeddings = None
clusters = None
document_metadata = []
processing_warnings = []
chroma_collection_name = None


#error handling

In [ ]:
from openai import APIError, RateLimitError, AuthenticationError

#low-level-functions (pdf convert, md splitting, clean, chunk)

In [ ]:
def clean(docs):
    cleaned_docs = []

    for doc in docs:
        text = doc.page_content

        # Remove HTML comments (picture placeholders)
        text = re.sub(
            r'<!-- Start of picture text -->.*?<!-- End of picture text -->',
            '',
            text,
            flags=re.DOTALL
        )

        # Remove HTML tags
        text = re.sub(r'<[^>]+>', ' ', text)

        # Remove markdown emphasis
        text = text.replace("**", "")
        text = text.replace("__", "")
        text = text.replace("_", "")

        # Normalize spaces
        text = re.sub(r'[ \t]+', ' ', text)

        # Collapse multiple blank lines
        text = re.sub(r'\n\s*\n+', '\n\n', text)

        doc.page_content = text.strip()
        cleaned_docs.append(doc)

    return cleaned_docs

In [ ]:
def chunk_documents(documents):
    chunks = recursive_splitter.split_documents(documents)
    return chunks

#functions for all_chunks and embeddings load

In [ ]:
def extract_documents(question_papers, progress):
    global document_metadata
    global processing_warnings

    extracted_chunks = []
    document_metadata = []
    processing_warnings = []
    total_files = len(question_papers)

    for i, pdf in enumerate(question_papers):
        source_name = os.path.basename(str(pdf))
        progress(i / total_files, desc=f"Extracting paper {i+1}/{total_files}: {source_name}")

        try:
            chunks = extract_pdf(pdf)

            if not chunks:
                raise RuntimeError("No usable chunks were produced.")

            for chunk in chunks:
                chunk.metadata["source"] = source_name

            first_metadata = chunks[0].metadata
            try:
                evidence = json.loads(first_metadata.get("metadata_evidence", "{}"))
            except Exception:
                evidence = {}

            document_metadata.append({
                "source": source_name,
                "class": first_metadata.get("class"),
                "subject": first_metadata.get("subject"),
                "year": first_metadata.get("year"),
                "exam_name": first_metadata.get("exam_name"),
                "metadata_source": first_metadata.get("metadata_source", "regex"),
                "metadata_evidence": evidence
            })

            extracted_chunks.extend(chunks)

        except Exception as e:
            warning = f"{source_name}: {type(e).__name__}: {e}"
            processing_warnings.append(warning)
            print(f"⚠️ Skipping paper: {warning}")

            # Do NOT abort the whole batch. Continue with the next PDF.
            progress(
                (i + 1) / total_files,
                desc=f"Skipped {source_name}; continuing with remaining papers..."
            )
            continue

    progress(1.0, desc="Finished processing uploaded PDFs.")
    return extracted_chunks


In [ ]:
def normalize_class(value):
    if not value:
        return None
    value = str(value).strip().upper()
    roman_map = {
        "I": "1", "II": "2", "III": "3", "IV": "4",
        "V": "5", "VI": "6", "VII": "7", "VIII": "8",
        "IX": "9", "X": "10", "XI": "11", "XII": "12"
    }
    return roman_map.get(value, value)


def extract_metadata_regex(text):
    """Extract Class/Subject/Year/Exam metadata from the document header only.

    Metadata is regex-only. Missing Class/Subject remains unverified; no LLM is used.
    """
    metadata = {"class": None, "subject": None, "year": None, "exam_name": None}

    # Inspect only the beginning of the extracted document.
    header = str(text or "")[:5000]
    lines = [re.sub(r"[*_`#|]+", " ", line).strip() for line in header.splitlines()]
    lines = [re.sub(r"\s+", " ", line) for line in lines if line.strip()]
    header_text = "\n".join(lines)

    # CLASS: explicit labels only.
    class_patterns = [
        r"\bCLASS\s*[-:]?\s*(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I|\d{1,2})\b",
        r"\bGRADE\s*[-:]?\s*(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I|\d{1,2})\b",
        r"\bSTANDARD\s*[-:]?\s*(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I|\d{1,2})\b",
        r"\bSTD\.?\s*[-:]?\s*(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I|\d{1,2})\b"
    ]
    for pattern in class_patterns:
        m = re.search(pattern, header_text, re.I)
        if m:
            metadata["class"] = normalize_class(m.group(1))
            break

    # SUBJECT: first look for explicit labels.
    aliases = {
        "Applied Mathematics": r"Applied\s+Mathematics",
        "Mathematics": r"Mathematics",
        "Physics": r"Physics",
        "Chemistry": r"Chemistry",
        "Biology": r"Biology",
        "Computer Science": r"Computer\s+Science",
        "Informatics Practices": r"Informatics\s+Practices",
        "English": r"English(?:\s+Language)?",
        "History": r"History",
        "Geography": r"Geography",
        "Economics": r"Economics",
        "Accountancy": r"Accountancy",
        "Business Studies": r"Business\s+Studies",
        "Political Science": r"Political\s+Science",
        "Sociology": r"Sociology",
        "Psychology": r"Psychology"
    }

    label_patterns = [
        r"\bSUBJECT(?:\s+NAME)?\s*[:\-–—]\s*(?P<value>[^\n|]+)",
        r"\bPAPER\s*[:\-–—]\s*(?P<value>[^\n|]+)"
    ]
    for pattern in label_patterns:
        m = re.search(pattern, header_text, re.I)
        if not m:
            continue
        value = re.sub(r"\s+", " ", m.group("value")).strip()
        for canonical, alias in sorted(aliases.items(), key=lambda x: len(x[0]), reverse=True):
            if re.search(rf"\b{alias}\b", value, re.I):
                metadata["subject"] = canonical
                break
        if metadata["subject"]:
            break

    # Fallback: accept a subject only when an entire short header line is the subject.
    # Do NOT search arbitrary body text.
    if not metadata["subject"]:
        for line in lines[:80]:
            candidate = re.sub(r"[^A-Za-z ]+", " ", line)
            candidate = re.sub(r"\s+", " ", candidate).strip()
            if not candidate or len(candidate) > 45:
                continue
            for canonical, alias in sorted(aliases.items(), key=lambda x: len(x[0]), reverse=True):
                if re.fullmatch(alias, candidate, re.I):
                    metadata["subject"] = canonical
                    break
            if metadata["subject"]:
                break

    # YEAR / EXAM: header only.
    m = re.search(r"\b(20\d{2})\b", header_text)
    if m:
        metadata["year"] = m.group(1)

    exam_patterns = [
        r"\b(Board Examination|Board Exam)\b",
        r"\b(Annual Examination|Annual Exam)\b",
        r"\b(Half Yearly Examination|Half Yearly Exam)\b",
        r"\b(Pre Board Examination|Pre Board Exam)\b",
        r"\b(Term I|Term II|Term 1|Term 2)\b"
    ]
    for pattern in exam_patterns:
        m = re.search(pattern, header_text, re.I)
        if m:
            metadata["exam_name"] = m.group(1)
            break

    return metadata


In [ ]:
def metadata_summary():
    """Generates a structured text summary of the current document metadata for the LLM's system prompt."""
    global document_metadata
    if not document_metadata:
        return "No document metadata available."

    summary = []
    for doc in document_metadata:
        summary.append(
            f"- File: {doc.get('source')}\n"
            f"  Class: {doc.get('class') or 'Unknown'}\n"
            f"  Subject: {doc.get('subject') or 'Unknown'}\n"
            f"  Year: {doc.get('year') or 'Unknown'}\n"
            f"  Exam Name: {doc.get('exam_name') or 'Unknown'}"
        )
    return "\n".join(summary)


def metadata_user_notice():
    """Generates a user-facing markdown notice if there are conflicting classes or subjects in the uploaded documents."""
    global document_metadata
    if not document_metadata:
        return ""

    classes = {doc.get('class') for doc in document_metadata if doc.get('class')}
    subjects = {doc.get('subject') for doc in document_metadata if doc.get('subject')}

    notices = []
    if len(classes) > 1:
        notices.append(f"multiple classes ({', '.join(sorted(classes))})")
    if len(subjects) > 1:
        notices.append(f"multiple subjects ({', '.join(sorted(subjects))})")

    if notices:
        return (
            "### ⚠️ Data Consistency Notice\n\n"
            f"The uploaded papers contain { ' and '.join(notices) }. "
            "Some topic frequencies and exam patterns may vary. "
            "The analysis has included all uploaded papers."
        )
    return ""

In [ ]:
def extract_pdf(pdf_path):
    try:
        md = pymupdf4llm.to_markdown(
            pdf_path,
            force_ocr=True,
            header=False,
            footer=False
        )
    except Exception as e:
        raise RuntimeError(f"PDF extraction failed for {os.path.basename(str(pdf_path))}: {e}") from e

    # 2. REGEX-ONLY METADATA EXTRACTION
    # Gemini is intentionally NOT used here. If Class or Subject cannot be
    # detected from the document text, it remains unverified.
    metadata = extract_metadata_regex(md)
    metadata["metadata_source"] = "regex"
    metadata["metadata_evidence"] = {}
    metadata["class"] = normalize_class(metadata.get("class"))

    print(f"Regex metadata for {os.path.basename(str(pdf_path))}: {metadata}")

    # 3. EXISTING MARKDOWN SPLITTING + CLEANING
    try:
        docs = markdown_splitter.split_text(md)
        docs = clean(docs)
    except Exception as e:
        raise RuntimeError(f"Markdown cleaning failed for {os.path.basename(str(pdf_path))}: {e}") from e

    # 4. EXISTING CHUNKING + METADATA
    try:
        chunks = chunk_documents(docs)
        if not chunks:
            raise RuntimeError("No usable chunks were produced.")

        for chunk in chunks:
            for key in ("class", "subject", "year", "exam_name"):
                value = metadata.get(key)
                if value is not None:
                    chunk.metadata[key] = str(value)
            chunk.metadata["metadata_source"] = "regex"

    except Exception as e:
        raise RuntimeError(f"Chunking failed for {os.path.basename(str(pdf_path))}: {e}") from e

    # Keep the metadata evidence field for compatibility with the existing
    # extract_documents() state collection. No AI verification is performed.
    chunks[0].metadata["metadata_evidence"] = json.dumps({})
    return chunks


In [ ]:
def load_embeddings():
    try:
        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        return embeddings

    except Exception as e:
        print(f"Error loading embedding model: {e}")
        return None

#making clusters and vector embeddings

In [ ]:
def cluster_documents(documents, embeddings, min_cluster_size=3):

    try:

        if not documents:
            return []

        if embeddings is None:
            return []

        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            metric="euclidean"
        )

        labels = clusterer.fit_predict(
            embeddings
        )

        clusters = defaultdict(list)

        for label, doc in zip(labels, documents):

            if label == -1:
                continue

            clusters[label].append(doc)

        return sorted(
            [
                {
                    "cluster_id": label,
                    "frequency": len(docs),
                    "documents": docs
                }
                for label, docs in clusters.items()
            ],
            key=lambda x: x["frequency"],
            reverse=True
        )

    except Exception as e:
        print(f"Clustering Error: {e}")
        return []

In [ ]:
def create_vectorstore(chunks, embedding_model):
    global chroma_collection_name

    try:
        # Give each analysis run its own collection so old uploads cannot
        # contaminate a new analysis while preserving persistent Chroma storage.
        chroma_collection_name = f"exam_papers_{int(time.time() * 1000)}"

        vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=embedding_model,
            collection_name=chroma_collection_name,
            persist_directory="./chroma_db"
        )

        return vectorstore

    except Exception as e:
        print(f"Error creating vector store: {e}")
        return None


In [ ]:
def query_vectorstore(vectorstore, query, k=5):
    try:
        results = vectorstore.similarity_search_with_score(
            query=query,
            k=k
        )
        return results

    except Exception as e:
        print(f"Error querying vector store: {e}")
        return None

#process_pipeline

In [ ]:
def process_documents(question_papers, progress=gr.Progress()):
    global vectorstore
    global all_chunks
    global embedding_model
    global embeddings
    global clusters
    global document_metadata
    global processing_warnings

    try:
        # Reset run-specific state before processing a new upload set.
        vectorstore = None
        all_chunks = None
        embedding_model = None
        embeddings = None
        clusters = None
        document_metadata = []
        processing_warnings = []

        # 1. Extract → Markdown → metadata → cleaning → chunking
        chunks = extract_documents(question_papers, progress)

        if not chunks:
            if processing_warnings:
                raise RuntimeError("No usable documents remained after skipping failed PDFs. See the processing warnings above.")
            raise RuntimeError("No documents could be extracted.")

        all_chunks = chunks

        # 2. Load embedding model
        progress(0.45, desc="Loading embedding model...")
        embedding_model = load_embeddings()

        if embedding_model is None:
            raise RuntimeError("Failed to load embedding model.")

        # 3. Generate embeddings
        progress(0.60, desc="Generating embeddings...")
        texts = [chunk.page_content for chunk in chunks]

        embeddings = embedding_model.embed_documents(texts)

        if not embeddings:
            raise RuntimeError("Failed to generate embeddings.")

        # 4. Create vector database
        progress(0.75, desc="Storing documents in ChromaDB...")
        vectorstore = create_vectorstore(chunks, embedding_model)

        if vectorstore is None:
            raise RuntimeError("Failed to create vector database.")

        # 5. Cluster documents
        progress(0.90, desc="Finding recurring patterns with HDBSCAN...")
        clusters = cluster_documents(all_chunks, embeddings)

        if processing_warnings:
            progress(1.0, desc=f"Processing complete with {len(processing_warnings)} skipped paper(s).")
            return f"Documents processed with {len(processing_warnings)} skipped paper(s)."

        progress(1.0, desc="Document processing complete.")
        return "Documents processed successfully."

    except Exception as e:
        # Never leave partially-created state available to the chatbot.
        vectorstore = None
        all_chunks = None
        embeddings = None
        clusters = None
        document_metadata = []
        processing_warnings = []
        embedding_model = None
        print(f"Processing Error: {type(e).__name__}: {e}")
        raise


#ai tool & system prompt

In [ ]:
INITIAL_ANALYSIS_PROMPT = """
You are an AI examination paper analyst.

Your task is to generate a polished, evidence-based report
from the uploaded examination papers.

You have access to:
- search_documents
- analyze_clusters

TOOL USAGE:
1. Use search_documents to inspect document metadata and retrieve
   actual questions and relevant content.
2. Use analyze_clusters to identify recurring question patterns.
3. Use both tools before preparing the final report.
4. If the first search is insufficient, perform additional searches.
5. Never expose tool calls, tool arguments, internal plans,
   or intermediate reasoning to the user.

METADATA VALIDATION:

Inspect class and subject metadata across all uploaded documents.

If different classes or subjects are detected, begin with:

⚠️ Data Consistency Notice

The uploaded papers contain different classes or subjects.
Some topic frequencies and exam patterns may vary.
The analysis will nevertheless include all uploaded papers.

If metadata is missing, explicitly mention which details
could not be verified.

Do not stop analysis due to metadata mismatches.

FINAL REPORT:

1. Repeated Questions
   - Exact repetitions
   - Similar questions
   - Years in which they appeared

2. Important Topics
   - Frequently tested topics
   - Evidence across papers

3. Recurring Concepts
   - Concepts tested in different ways

4. Types of Questions
   - MCQs
   - Short answers
   - Long answers
   - Numericals
   - Derivations
   - Comprehension
   - Writing tasks
   - Other observed formats

5. Exam Pattern
   - Marks distribution
   - Question structure
   - Time allocation
   - Recurring sections

6. Preparation Insights
   - Observed recurring concepts
   - Areas requiring practice
   - Patterns across available years

EVIDENCE RULES:
- Do not invent questions, marks, years, or frequencies.
- Do not claim that a topic is repeated unless supported by documents.
- Distinguish exact repetition from similar question types.
- State when evidence is insufficient.
- Do not predict guaranteed future questions.

OUTPUT RULE:

Return ONLY the final user-facing analysis.

Never output:
- "Here's my plan"
- "Let's start by calling..."
- Tool-use narration
- Internal reasoning
- JSON tool arguments

Begin directly with the report heading or metadata warning.
"""

In [ ]:
system_prompt = """
# ROLE
You are an AI Exam Analyzer specialized in analyzing uploaded previous-year examination papers.

# SOURCE OF TRUTH
The uploaded papers and their retrieved document evidence are the ONLY source of factual claims about the examinations.
Never use general subject knowledge to fill missing evidence.
Never invent questions, topics, marks, sections, frequencies, repetitions, or patterns.

# AVAILABLE TOOLS
## search_documents
Use this to retrieve actual text from the uploaded papers. Prefer this for verifying exact questions, sections, marks, question types, and document content.

## analyze_clusters
Use this only as a secondary aid to locate related document evidence. Its grouping is not proof of an exact repeated question and its internal frequency is not a count of exam questions.

# STRICT EVIDENCE RULE
If a claim cannot be supported by actual paper text/evidence available in the current conversation, DO NOT make the claim.
Do not replace missing evidence with phrases such as "likely", "typically", "generally", "usually", "may include", or "expected".
Those words must not be used to speculate about what the uploaded papers contain.

# CROSS-PAPER RULE
A pattern is recurring only if the evidence shows it in more than one relevant paper.
If subjects/classes differ, analyze them separately. Never merge Chemistry, Mathematics, English, or other subject-specific findings merely because they were uploaded together.
If a topic appears in only one paper, call it an observed topic rather than a recurring/frequently tested topic.

# REPETITION RULE
Exact repetition requires substantially matching actual question content from multiple papers.
Conceptual similarity requires evidence that different questions test the same underlying concept.
Semantic grouping alone is insufficient to claim an exact repetition.

# METADATA RULES
Class and Subject come only from the application's regex metadata extraction.
Do not infer missing Class or Subject from filenames, question content, or general knowledge.
Do not mention Year or Exam Name unless the user asks for them.

# RESPONSE STYLE
Produce a polished examination-analysis report for the student.
Never mention tools, tool calls, retrieval, clusters, cluster IDs, cluster frequency, embeddings, ChromaDB, vector databases, or internal processing.
Never expose raw internal metadata tables.
Do not give generic textbook advice; preparation insights must be tied to observed paper evidence.
Do not make predictions or guarantees about future examinations.
"""


In [ ]:
def analyze_clusters(top_n=5):
    global clusters

    try:
        if clusters is None:
            return "No documents have been processed."

        if not clusters:
            return "No meaningful related-document evidence was found."

        output = []

        for cluster in clusters[:top_n]:
            # Do not expose cluster frequency to the LLM.
            # Recurrence must be verified from actual document evidence.
            docs = cluster.get("documents", [])[:8]
            if not docs:
                continue
            output.append(
                "Related document evidence:\n"
                + "\n\n---\n\n".join(doc.page_content for doc in docs)
            )

        return "\n\n========================\n\n".join(output) if output else "No usable related-document evidence was found."

    except Exception as e:
        print(f"Cluster Analysis Error: {e}")
        return "Unable to analyze related document evidence."


In [ ]:
def search_documents(query, k=5):
    """Search for relevant chunks in the vector store matching the query."""
    global vectorstore
    if vectorstore is None:
        return "No documents have been loaded or processed yet. Please upload files first."

    try:
        results = query_vectorstore(vectorstore, query, k=k)
        if not results:
            return "No matching results found in the documents."

        formatted_results = []
        for doc, score in results:
            source = doc.metadata.get("source", "Unknown Source")
            class_val = doc.metadata.get("class", "Unknown")
            subject_val = doc.metadata.get("subject", "Unknown")

            formatted_results.append(
                f"[Source: {source} | Class: {class_val} | Subject: {subject_val} | Similarity Score: {score:.4f}]\n"
                f"{doc.page_content}"
            )
        return "\n\n---\n\n".join(formatted_results)
    except Exception as e:
        return f"Error searching documents: {str(e)}"

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_documents",
            "description":
            "To search from uploaded exam documents to find questions, topics, and patterns.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description":
                        "Topic, subject, or question to search"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_clusters",
            "description": "Analyze uploaded question papers to find the most frequently repeated question patterns and concepts. Use this when the user asks for most asked questions, repeated questions, important questions, frequent topics, or exam pattern analysis.",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    }
]

#llms

In [ ]:
def _llm_call_with_fallback(messages, model, tools=None, response_format=None):
    """Call client1 first and switch to client2 on rate limiting."""
    kwargs = {
        "model": model,
        "messages": messages
    }

    if tools is not None:
        kwargs["tools"] = tools

    if response_format is not None:
        kwargs["response_format"] = response_format

    try:
        return client1.chat.completions.create(**kwargs), client1
    except RateLimitError:
        print("Primary API limit reached. Switching to Secondary...")
        return client2.chat.completions.create(**kwargs), client2


def ask_llm(query, history):
    start_time = time.time()

    metadata_context = metadata_summary()

    messages = [
        {
            "role": "system",
            "content": (
                system_prompt
                + "\n\nCURRENT UPLOAD METADATA:\n"
                + metadata_context
            )
        }
    ]

    messages.extend(history)
    messages.append({"role": "user", "content": query})

    model = "gemini-2.5-flash-lite"
    active_client = client1

    try:
        response, active_client = _llm_call_with_fallback(
            messages,
            model,
            tools=tools
        )
        message = response.choices[0].message

        while message.tool_calls:
            # The assistant message must contain ALL tool calls together.
            assistant_tool_calls = []
            tool_results = []

            for tool_call in message.tool_calls:
                name = tool_call.function.name
                raw_arguments = tool_call.function.arguments or "{}"

                try:
                    args = json.loads(raw_arguments)
                except json.JSONDecodeError:
                    args = {}

                if name == "search_documents":
                    tool_result = search_documents(args.get("query", ""))
                elif name == "analyze_clusters":
                    tool_result = analyze_clusters(args.get("top_n", 5))
                else:
                    tool_result = f"Unknown tool: {name}"

                assistant_tool_calls.append({
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": name,
                        "arguments": raw_arguments
                    }
                })

                tool_results.append({
                    "tool_call_id": tool_call.id,
                    "content": str(tool_result)
                })

            messages.append({
                "role": "assistant",
                "tool_calls": assistant_tool_calls
            })

            for result in tool_results:
                messages.append({
                    "role": "tool",
                    "tool_call_id": result["tool_call_id"],
                    "content": result["content"]
                })

            # Fallback also applies to subsequent tool-calling rounds.
            response, active_client = _llm_call_with_fallback(
                messages,
                model,
                tools=tools
            )
            message = response.choices[0].message

        print(f"LLM response time: {time.time() - start_time:.2f}s")
        return message.content or "The model returned no text."

    except AuthenticationError:
        return "API key error. Please check your Gemini API configuration."

    except RateLimitError:
        return "Both API clients have reached their rate limits. Please try again later."

    except APIError as e:
        print(f"API Error: {e}")
        return "The AI service is temporarily unavailable."

    except Exception as e:
        print(f"Unexpected Error: {e}")
        return "Sorry, I could not process your request."


#User interface

In [ ]:
def _disabled_controls():
    # Must match the exact output order of process_and_analyze().
    return (
        gr.update(interactive=False),  # message
        gr.update(interactive=False),  # send
        gr.update(interactive=False),  # repeated
        gr.update(interactive=False),  # topics
        gr.update(interactive=False),  # concepts
        gr.update(interactive=False),  # pattern
        gr.update(interactive=False),  # clear chat
        False                          # ready_state
    )


def _enabled_controls():
    return (
        gr.update(interactive=True),
        gr.update(interactive=True),
        gr.update(interactive=True),
        gr.update(interactive=True),
        gr.update(interactive=True),
        gr.update(interactive=True),
        gr.update(interactive=True),
        True
    )


def initial_analysis():
    # The prompt explicitly requires both tools. Metadata is also supplied
    # through the current-upload context inside ask_llm().
    return ask_llm(
        INITIAL_ANALYSIS_PROMPT,
        []
    )


def process_and_analyze(files, progress=gr.Progress()):
    global processing_warnings
    # Every yield MUST contain exactly 10 values matching the 10 Gradio outputs:
    # status, chatbot, message, send, repeated, topics, concepts, pattern,
    # clear_chat, ready_state.
    disabled = _disabled_controls()

    if not files:
        yield (
            "⚠️ Please upload PDF papers first.",
            [],
            *disabled
        )
        return

    try:
        # Clear stale UI/state before a new run.
        yield (
            "⏳ Starting document processing...",
            [],
            *disabled
        )

        yield (
            "📄 Extracting PDF text with PyMuPDF4LLM and detecting metadata...",
            [],
            *disabled
        )

        # Pass Gradio's live Progress object into the existing pipeline.
        process_result = process_documents(files, progress=progress)

        if processing_warnings:
            skipped = "\n".join(f"• {warning}" for warning in processing_warnings)
            yield (
                "⚠️ Some papers could not be processed and were skipped. Continuing with the usable papers...\n\n" + skipped,
                [],
                *disabled
            )

        yield (
            "🧹 Cleaning Markdown, chunking documents, and preparing embeddings...",
            [],
            *disabled
        )

        yield (
            "🧠 Embeddings, ChromaDB storage, and HDBSCAN clustering completed. "
            "Checking extracted document metadata...",
            [],
            *disabled
        )

        metadata_notice = metadata_user_notice()

        yield (
            "🔎 Document processing complete. Running evidence-based initial analysis...",
            [],
            *disabled
        )

        # The model receives metadata internally through ask_llm(); the report
        # itself must remain a clean professional analysis.
        report = initial_analysis()

        if metadata_notice:
            report = report.rstrip() + "\n\n" + metadata_notice

        chat_history = [
            {
                "role": "assistant",
                "content": report
            }
        ]

        enabled = _enabled_controls()

        if processing_warnings:
            success_status = (
                f"🟢 Analysis complete with {len(processing_warnings)} skipped paper(s). "
                "The chatbot and quick analysis are ready."
            )
        else:
            success_status = "🟢 Analysis complete! Chatbot and quick analysis are ready."

        yield (
            success_status,
            chat_history,
            *enabled
        )

    except Exception as e:
        # Catch extraction, metadata, embedding, ChromaDB, clustering,
        # initial-analysis, and Gradio-callback exceptions. Never unlock UI.
        print(f"Processing error: {type(e).__name__}: {e}")

        # Clear backend state as well so a failed run cannot reuse partial data.
        try:
            vectorstore = None
            all_chunks = None
            embeddings = None
            clusters = None
            document_metadata = []
            processing_warnings = []
        except Exception as cleanup_error:
            print(f"State cleanup error: {cleanup_error}")

        yield (
            f"❌ Processing failed: {type(e).__name__}: {e}",
            [],
            *_disabled_controls()
        )


def chat_response(message, history, ready):
    if not ready:
        return history, ""

    if not message or not message.strip():
        return history, ""

    try:
        response = ask_llm(message, history)
    except Exception as e:
        print(f"Chat error: {type(e).__name__}: {e}")
        response = f"❌ Chat request failed: {type(e).__name__}: {e}"

    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response}
    ]

    return history, ""


def quick_action(prompt, history, ready):
    if not ready:
        return history

    try:
        response = ask_llm(prompt, history)
    except Exception as e:
        print(f"Quick analysis error: {type(e).__name__}: {e}")
        response = f"❌ Quick analysis failed: {type(e).__name__}: {e}"

    return history + [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": response}
    ]


def clear_chat():
    return []


In [ ]:
with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="indigo",
        neutral_hue="slate"
    ),
    title="AI Exam Analyzer"
) as demo:

    ready_state = gr.State(False)

    gr.Markdown(
        """
        # 🎓 ExamScope

        ### -An AI-powered exam analysis tool designed to help students prepare smarter. By analyzing provided question papers, it identifies recurring questions, important topics, question patterns, and examination trends, helping students focus their preparation and make more informed study decisions.

        Upload previous-year papers. Papers from the same class and subject
        are recommended*.
        """
    )

    with gr.Row():
        with gr.Column(scale=3):
            gr.Markdown("### 📄 Question Papers")

            pdf_files = gr.File(
                label="Drop your PDF papers here",
                file_types=[".pdf"],
                file_count="multiple",
                type="filepath"
            )

            analyze_btn = gr.Button(
                "✨ Analyze Question Papers",
                variant="primary",
                size="lg"
            )

        with gr.Column(scale=1):
            gr.Markdown("### Status")

            status = gr.Textbox(
                value="🔵 Waiting for papers",
                label="Processing Status",
                interactive=False,
                lines=4
            )

    gr.Markdown("## 🤖 Exam Assistant")

    chatbot = gr.Chatbot()

    with gr.Row():
        repeated_btn = gr.Button(
            "🔁 Repeated Questions",
            interactive=False
        )

        topics_btn = gr.Button(
            "📖 Important Topics",
            interactive=False
        )

        concepts_btn = gr.Button(
            "🧠 Recurring Concepts",
            interactive=False
        )

        pattern_btn = gr.Button(
            "📊 Exam Pattern",
            interactive=False
        )

    with gr.Row():
        message = gr.Textbox(
            placeholder="Ask something about your uploaded papers...",
            show_label=False,
            scale=5,
            interactive=False
        )

        send_btn = gr.Button(
            "➤",
            variant="primary",
            scale=1,
            interactive=False
        )

    clear_btn = gr.Button(
        "🗑️ Clear Chat",
        interactive=False
    )

    # Process documents and generate the initial report.
    analyze_btn.click(
        fn=process_and_analyze,
        inputs=[pdf_files],
        outputs=[
            status,
            chatbot,
            message,
            send_btn,
            repeated_btn,
            topics_btn,
            concepts_btn,
            pattern_btn,
            clear_btn,
            ready_state
        ]
    )

    # Normal chatbot.
    send_btn.click(
        fn=chat_response,
        inputs=[message, chatbot, ready_state],
        outputs=[chatbot, message]
    )

    message.submit(
        fn=chat_response,
        inputs=[message, chatbot, ready_state],
        outputs=[chatbot, message]
    )

    # Quick analysis buttons.
    repeated_btn.click(
        fn=lambda h, r: quick_action(
            "Find exact and conceptually similar repeated questions across all uploaded papers. "
            "Use both search_documents and analyze_clusters where appropriate. Include supporting document evidence.",
            h, r
        ),
        inputs=[chatbot, ready_state],
        outputs=chatbot
    )

    topics_btn.click(
        fn=lambda h, r: quick_action(
            "Identify important and frequently tested topics across all uploaded papers, "
            "with supporting document evidence. Use both available tools.",
            h, r
        ),
        inputs=[chatbot, ready_state],
        outputs=chatbot
    )

    concepts_btn.click(
        fn=lambda h, r: quick_action(
            "Identify recurring concepts and explain how they are tested in different ways. "
            "Use supporting document evidence.",
            h, r
        ),
        inputs=[chatbot, ready_state],
        outputs=chatbot
    )

    pattern_btn.click(
        fn=lambda h, r: quick_action(
            "Analyze the exam pattern, question types, marks distribution, and recurring sections. "
            "Report only what the uploaded papers support.",
            h, r
        ),
        inputs=[chatbot, ready_state],
        outputs=chatbot
    )

    clear_btn.click(
        fn=clear_chat,
        outputs=chatbot
    )

    gr.Markdown (
        """Some document details could not be identified or verified from the uploaded PDFs. This may be due to missing or unclear information in the source documents. The analysis will proceed using the available content, although some results may be less precise."""
        )
demo.queue()
demo.launch()
